# TÓPICOS DE MATEMÁTICA PARA SISTEMAS AUTÓNOMOS (TOMSA)
## TRABALHO LETIVO - ENTREGA 1

---
### GRUPO
* **N:** 1211756 **Nome:** Geovani Lopes
* **N:** 1120542 **Nome:** Luis Lei
* **N:** 1261726 **Nome:** Hugo Amado 
---
### Conteúdo
1. [Identificação dos objetivos e estruturação do problema](#1-identificação-dos-objetivos-e-estruturação-do-problema)
2. [Justificação da relevância das bibliotecas Python selecionadas](#2-justificação-da-relevância-das-bibliotecas-python-selecionadas)
3. [Metodologia proposta para o desenvolvimento do projeto](#3-metodologia-proposta-para-o-desenvolvimento-do-projeto)


## 1. Identificação dos objetivos e estruturação do problema
[De volta para o Conteúdo](#conteúdo)

### 1.1 Objetivo Geral
O presente trabalho tem como objetivo projetar, desenvolver e validar rigorosamente uma aplicação computacional em **Python** destinada à visualização, sincronização temporal e análise cinemática de trajetórias de múltiplos robôs autónomos de superfície e submarinos (ASVs/AUVs). O sistema baseia-se em telemetria real obtida pelo **INESC TEC** durante campanhas marinhas realizadas na costa da **Aguçadoura** (Póvoa de Varzim, Portugal).

A aplicação realiza a conversão de coordenadas geodésicas de navegação (WGS-84: latitude, longitude, altitude e quaternions de atitude) para referenciais cartesianos locais tangentes (**NED** - *North-East-Down* e **ENU** - *East-North-Up*), projeta as posições relativas e atitudes no referencial próprio do corpo (**Body Frame**) do veículo selecionado, e calcula grandezas esféricas relativas de visada (**AER** - *Azimuth, Elevation, Range*).

### 1.2 Objetivos Específicos e Requisitos do Sistema
| ID | Funcionalidade | Descrição e Critério de Aceitação |
|:--:|:---|:---|
| **O1** | **Visão Global** | Apresentação de todas as trajetórias e posições instantâneas dos robôs num referencial cartesiano comum (NED/ENU) centrado na origem da missão, com diferenciação cromática e vetores de proa (*heading*). |
| **O2** | **Visão Local** | Representação dinâmica da vizinhança centrada no robô selecionado (origem $(0,0,0)$ orientada com a sua proa/eixo longitudinal), exibindo robôs vizinhos. |
| **O3** | **Controlo temporal** | Barra deslizante (*slider*), botões *Play/Pause* e saltos temporais, operando sobre uma série temporal interpolada e sincronizada (interpolação linear para posições e SLERP para atitudes). |
| **O4** | **Configuração YAML** | Definição declarativa e modular de parâmetros da missão: coordenadas da origem geodésica, lista de ficheiros CSV, cores, robô selecionado, limiar de proximidade e parâmetros de replay. |
| **O5** | **Registo de saida** | Registo estruturado de eventos de execução (*logging*), exportação de telemetria sincronizada em CSV, GeoJSON (para visualização SIG/Google Earth) e métricas globais em JSON (incluindo cálculo de CPA - *Closest Point of Approach*). |
| **O6** | **Visão 3D em primeira pessoa** | Implementação de raiz (*first-principles*) de todas as fórmulas de conversão geodésica e cinemática com NumPy, com validação cruzada exaustiva contra bibliotecas padrão (*pymap3d* e *pyproj*). |

### 1.3 Modelação Matemática do Problema

#### A. Conversão Geodésica para ECEF (Elipsoide WGS-84)
Seja $\varphi$ a latitude geodésica, $\lambda$ a longitude e $h$ a altitude elipsoidal. O raio de curvatura normal no primeiro vertical é dado por:
$$N(\varphi) = \frac{a}{\sqrt{1 - e^2 \sin^2 \varphi}}$$
onde $a = 6\,378\,137.0\text{ m}$ (semieixo maior) e $e^2 = 2f - f^2 \approx 0.00669438$ (primeira excentricidade ao quadrado, com $f = 1/298.257223563$). As coordenadas cartesianas ECEF são expressas por:
$$X = (N(\varphi) + h) \cos \varphi \cos \lambda$$
$$Y = (N(\varphi) + h) \cos \varphi \sin \lambda$$
$$Z = \left(N(\varphi)(1 - e^2) + h\right) \sin \varphi$$

#### B. Conversão ECEF para Plano Tangente Local (ENU e NED)
Dada uma origem geodésica $(\varphi_0, \lambda_0, h_0)$ com coordenadas ECEF $\mathbf{r}_{0} = (X_0, Y_0, Z_0)^T$, o vetor de translação relativo é $\Delta\mathbf{r} = \mathbf{r}_{ECEF} - \mathbf{r}_0$. A rotação ortogonal para o referencial ENU é obtida por:
$$\begin{bmatrix} x_E \\ y_N \\ z_U \end{bmatrix} = \begin{bmatrix} -\sin\lambda_0 & \cos\lambda_0 & 0 \\ -\sin\varphi_0 \cos\lambda_0 & -\sin\varphi_0 \sin\lambda_0 & \cos\varphi_0 \\ \cos\varphi_0 \cos\lambda_0 & \cos\varphi_0 \sin\lambda_0 & \sin\varphi_0 \end{bmatrix} \begin{bmatrix} X - X_0 \\ Y - Y_0 \\ Z - Z_0 \end{bmatrix}$$
Para o referencial náutico e aeronáutico **NED** (*North-East-Down*):
$$x_N = y_N^{\text{ENU}}, \quad y_E = x_E^{\text{ENU}}, \quad z_D = -z_U^{\text{ENU}}$$

#### C. Sistema Esférico Local AER (Azimute, Elevação, Range)
A partir das coordenadas cartesianas locais:
$$\text{Range} = \sqrt{x_N^2 + y_E^2 + z_D^2}$$
$$\text{Azimuth} = \operatorname{atan2}(y_E, x_N) \pmod{360^\circ}$$
$$\text{Elevation} = \operatorname{atan2}(-z_D, \sqrt{x_N^2 + y_E^2})$$

#### D. Orientação por Quaternions e Transformação para o Referencial do Corpo (Body Frame)
A atitude do veículo no referencial NED é descrita pelo quaternion unitário $\mathbf{q} = (q_x, q_y, q_z, q_w)$. A matriz de rotação $\mathbf{R}_{B/NED}$ (que mapeia vetores do corpo para o NED) é:
$$\mathbf{R}_{B/NED} = \begin{bmatrix} 1 - 2(q_y^2 + q_z^2) & 2(q_x q_y - q_z q_w) & 2(q_x q_z + q_y q_w) \\ 2(q_x q_y + q_z q_w) & 1 - 2(q_x^2 + q_z^2) & 2(q_y q_z - q_x q_w) \\ 2(q_x q_z - q_y q_w) & 2(q_y q_z + q_x q_w) & 1 - 2(q_x^2 + q_y^2) \end{bmatrix}$$
A posição relativa de um robô vizinho $j$ no referencial próprio do robô observador $i$ é obtida pela translação e projeção inversa:
$$\mathbf{p}_j^{\text{Body}(i)} = \mathbf{R}_{B/NED}^T \left(\mathbf{p}_j^{NED} - \mathbf{p}_i^{NED}\right) = \begin{bmatrix} x_{\text{proa}} \\ y_{\text{estibordo}} \\ z_{\text{quilha}} \end{bmatrix}$$


## 2. Justificação da relevância das bibliotecas Python selecionadas
[De volta para o Conteúdo](#conteúdo)

A seleção de ferramentas baseou-se em critérios de desempenho computacional, maturidade científica, robustez numérica e facilidade de integração:

| Biblioteca | Papel no Projeto | Justificação Técnica e Vantagens |
|:---|:---|:---|
| **NumPy** | Vetorização e Álgebra Linear | Núcleo de cálculo numérico em C. Permite calcular conversões de dezenas de milhares de pontos geodésicos e multiplicações matriciais $3 \times 3$ simultâneas sem ciclos lentos em Python puro. |
| **pandas** | Leitura e Manipulação Tabular | Otimizada para leitura de ficheiros CSV com inferência de tipos (`dtypes`), filtragem, ordenação temporal e junção de telemetrias com diferentes frequências de amostragem. |
| **Matplotlib** | Visualização Gráfica e GUI Interativa | Permite controlo absoluto da renderização gráfica em 2D/3D, exportação de figuras vetoriais de alta resolução (PDF) e construção de controlos GUI interativos nativos (`Slider`, `Button`, `RadioButtons`) sem dependências pesadas de frontend web. |
| **PyYAML** | Configuração Declarativa | Leitura segura (`yaml.safe_load`) de parâmetros externos de missão, permitindo alterar ficheiros de dados, origens, cores e limiares sem modificar o código-fonte. |
| **SciPy** | Rotações Espaciais e Interpolação | Disponibiliza algoritmos otimizados de interpolação esférica linear (`Slerp`) para quaternions e álgebra espetral (`eigh`) para extração de semi-eixos de elipses de incerteza a partir da covariância. |
| **Plotly** | Visualizador 3D Interativo (Body Frame) | Renderização acelerada por GPU (WebGL) no navegador com câmara 3D em 1ª pessoa, suporte a atalhos de teclado e exportação para HTML autónomo. |
| **pymap3d** | Validação Numérica de Referência | Biblioteca padrão de referência geodésica em Python (utilizada pela comunidade aeroespacial/marítima), servindo de *ground truth* para certificar as fórmulas de raiz implementadas com erro $< 10^{-6}\text{ m}$. |
| **pyproj** | Verificação Geodésica Independente | Interface para a biblioteca cartográfica internacional PROJ, permitindo uma segunda camada de validação cruzada independente de coordenadas elipsoidais. |
| **pytest** | Automação de Testes de Regressão | Framework de testes unitários parametrizados, garantindo a verificação contínua e impedindo regressões em todo o pipeline matemático. |

> **Decisão Arquitetural:** Todas as conversões de coordenadas (Geodésico $\leftrightarrow$ ECEF $\leftrightarrow$ NED/ENU $\leftrightarrow$ AER) e transformações de referencial do corpo são implementadas **de raiz pelo grupo** com recurso a NumPy, comprovando o domínio matemático. As bibliotecas `pymap3d` e `pyproj` são utilizadas única e exclusivamente como autoridade de teste e validação cruzada nos testes unitários.

## 3. Metodologia proposta para o desenvolvimento do projeto
[De volta para "Conteúdo"](#Conteúdo)

Para a execução do projeto, adota-se a metodologia ágil Scrum em conjunto com práticas de garantia de qualidade:

- Gestão Iterativa (Scrum): o trabalho será faseado em sprints, permitindo a validação progressiva das funcionalidades e uma adaptação rápida a imprevistos.
- Testes Unitários Contínuos (pytest): serão definidos testes automáticos para cada módulo desenvolvido, com foco especial na exatidão das fórmulas cinemáticas, interpolações e transformações de coordenadas.   

